In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# -----------------------------
# 1. Configuration
# -----------------------------
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

Using device: cuda


In [2]:


# -----------------------------
# 2. Load and preprocess data
# -----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts image to tensor in [0,1]
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

full_train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

# Split training set into training + validation
train_size = int(0.7 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------
# 3. Define the neural network
# -----------------------------
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),          # 28x28 -> 784
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)     # 10 output classes: 0, 1, 2, 3, 4, .. 9
        )

    def forward(self, x):
        return self.model(x)

model = MLP().to(DEVICE)

# -----------------------------
# 4. Loss function and optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# -----------------------------
# 5. Training function
# -----------------------------
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# -----------------------------
# 6. Evaluation function
# -----------------------------
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# -----------------------------
# 7. Training loop
# -----------------------------
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

    print(f"Epoch [{epoch+1}/{EPOCHS}]")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

# -----------------------------
# 8. Final test evaluation
# -----------------------------
test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)
print(f"\nTest Loss: {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")

# -----------------------------
# 9. Save the model
# -----------------------------
torch.save(model.state_dict(), "mnist_mlp.pth")
print("Model saved as mnist_mlp.pth")

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 580kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.37MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.50MB/s]


Epoch [1/5]
  Train Loss: 0.2693 | Train Acc: 0.9176
  Val   Loss: 0.1340 | Val   Acc: 0.9587
Epoch [2/5]
  Train Loss: 0.1050 | Train Acc: 0.9674
  Val   Loss: 0.1146 | Val   Acc: 0.9652
Epoch [3/5]
  Train Loss: 0.0732 | Train Acc: 0.9763
  Val   Loss: 0.1056 | Val   Acc: 0.9676
Epoch [4/5]
  Train Loss: 0.0555 | Train Acc: 0.9823
  Val   Loss: 0.0960 | Val   Acc: 0.9728
Epoch [5/5]
  Train Loss: 0.0380 | Train Acc: 0.9880
  Val   Loss: 0.0996 | Val   Acc: 0.9732

Test Loss: 0.0890 | Test Accuracy: 0.9761
Model saved as mnist_mlp.pth
